In [1]:
import pandas as pd

df_full_dataset = pd.read_csv("../data/clean/merged_data_sets.csv",
                              index_col=0, parse_dates=["date_time"],
                              dtype={"Variation": "string"})
df_full_dataset.head()

,client_id,Variation,visitor_id,visit_id,process_step,date_time,client_tenure_yr,client_tenure_mnth,client_age,gender,num_accts,total_balance,calls_6_mnth,logons_6_mnth
0,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
1,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
2,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
3,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
4,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0


In [2]:
df_full_dataset.duplicated().sum()

np.int64(5934)

In [3]:
# keeps the first, drops the copies
df_full_dataset = df_full_dataset.drop_duplicates()
df_full_dataset.duplicated().sum()

np.int64(0)

In [4]:
df_full_dataset.head()

,client_id,Variation,visitor_id,visit_id,process_step,date_time,client_tenure_yr,client_tenure_mnth,client_age,gender,num_accts,total_balance,calls_6_mnth,logons_6_mnth
0,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
1,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
2,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
3,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
4,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0


In [5]:
df_full_dataset["Variation"].value_counts(dropna=False)

df_test    = df_full_dataset[df_full_dataset["Variation"] == "Test"]
df_control = df_full_dataset[df_full_dataset["Variation"] == "Control"]

df_test["client_id"].nunique(), df_control["client_id"].nunique()

(26968, 23532)

KPIs
1. Completion rate: the percentage of clients in each group who reached confirm.
- Perfect completed rate: Clients that completed the 5 steps (start, step 1, step 2, step 3, confirm) only 1 time each
- Completed rate: Clients that completed the 5 steps even if they repeated once or more
- Error Completion: Clients that have the 5 steps but more than 1 count in at least 1 step
- Not completed rate: Clients that did not complete the 5 steps (Here we consider that 1 step missing is an error)

- Decide what counts as a valid completion. Does a confirm without a start count?

In [6]:
# Keep only the experiment groups
df_exp = df_full_dataset[df_full_dataset["Variation"].isin(["Test", "Control"])]

# True/False column per step ("did this client ever reach it?")
client_steps = (
    pd.crosstab([df_exp["client_id"], df_exp["Variation"]], df_exp["process_step"])
      .gt(0)
      .reset_index()
      .rename_axis(columns=None)
)
client_steps.head(10)

,client_id,Variation,confirm,start,step_1,step_2,step_3
0,555,Test,True,True,True,True,True
1,647,Test,True,True,True,True,True
2,934,Test,False,True,False,False,False
3,1028,Control,False,True,True,True,True
4,1104,Control,False,True,False,False,False
5,1186,Control,False,True,True,True,False
6,1195,Control,True,True,True,True,True
7,1197,Control,True,True,True,True,True
8,1336,Test,True,True,True,True,True
9,1346,Test,False,True,True,True,True


In [7]:
# Clients who confirmed but never had a start
confirm_no_start = client_steps["confirm"] & ~client_steps["start"]
confirm_no_start.groupby(client_steps["Variation"]).sum()

Variation
Control    116
Test       275
dtype: int64

In [ ]:
# fazer isto para os que clicaram só confimed
# Clients who confirmed but never had a start
# confirm_no_start = client_steps["confirm"] & ~client_steps["start"]
# confirm_no_start.groupby(client_steps["Variation"]).sum()

In [8]:
steps = ["start", "step_1", "step_2", "step_3", "confirm"]

# One row per client, number of times each step was recorded
client_counts = (
    pd.crosstab([df_exp["client_id"], df_exp["Variation"]], df_exp["process_step"])
      .reset_index()
      .rename_axis(columns=None)
)

# Classify each client

# Every step exactly once
client_counts["perfect_completed"] = (client_counts[steps] == 1).all(axis=1)
# Every step at least once
client_counts["completed"] = (client_counts[steps] >= 1).all(axis=1)
# Missing at least one step
client_counts["not_completed"] = ~client_counts["completed"]
# Completed, but with repeats/errors (except perfect_completed)
client_counts["error_completed"] = client_counts["completed"] & ~client_counts["perfect_completed"]

In [9]:
cols = ["perfect_completed", "completed", "not_completed", "error_completed"]

# Number of clients
completion_counts = client_counts.groupby("Variation")[cols].sum()
completion_counts["total_clients"] = client_counts.groupby("Variation")["client_id"].nunique()
completion_counts

,perfect_completed,completed,not_completed,error_completed,total_clients
Variation,,,,,
Control,7360,15206,8326,7846,23532
Test,8273,18338,8630,10065,26968


In [10]:
# Percentage of clients
completion_pct = client_counts.groupby("Variation")[cols].mean().mul(100).round(2)
completion_pct

,perfect_completed,completed,not_completed,error_completed
Variation,,,,
Control,31.28,64.62,35.38,33.34
Test,30.68,68.00,32.00,37.32


2. Step-by-step funnel: the percentage of clients reaching each step

- start → step_1 → step_2 → step_3 → confirm

- For the completed rate
    - Which group returns to a step more often
    - Which step is more frequent to return to
- For the not completed clients
    - At which step clients stop the process most often (the furthest step they reached before abandoning)
    - Whether drop-off happens mostly early (start / step_1) or late (step_2 / step_3), to understand if clients give up in the first steps
    - Clients who reached confirm but skipped a step are kept as a separate category (“reached confirm with a gap”), since they did not abandon the process

In [11]:
completed = client_counts[client_counts["completed"]].copy()

# Extra visits per step: a step seen 3 times = 2 returns; seen once = 0
returns = (completed[steps] - 1) #.clip(lower=0)
completed["total_returns"] = returns.sum(axis=1)

In [12]:
group_returns = completed.groupby("Variation").agg(
    completed_clients = ("client_id", "nunique"),
    clients_with_returns = ("error_completed", "sum"),
    avg_returns_per_client = ("total_returns", "mean"),
)

group_returns["pct_with_returns"] = (
    group_returns["clients_with_returns"] / group_returns["completed_clients"] * 100
).round(2)

group_returns["avg_returns_per_client"] = group_returns["avg_returns_per_client"].round(2)
group_returns

,completed_clients,clients_with_returns,avg_returns_per_client,pct_with_returns
Variation,,,,
Control,15206,7846,2.05,51.60
Test,18338,10065,2.30,54.89


Returns among completed clients
- About half of the clients who completed repeated at least one step: 51.60% Control vs 54.89% Test.
- Test completers also repeat slightly more often: 2.30 vs 2.05 extra step visits per client.
- The new design gets more clients to the end, but not through a smoother path.
- A repeated step is not necessarily an error (clients may go back on purpose), so it is treated as possible friction.

In [13]:
# Total number of returns per step
step_return_counts = returns.groupby(completed["Variation"]).sum().T # .T flips the table so the steps are rows and Test/Control are columns
step_return_counts

#remover os que tem apenas um step de cada

Variation,Control,Test
start,12751,16065
step_1,6767,9967
step_2,5568,6055
step_3,4258,3668
confirm,1819,6340


Total returns per step
- Start has by far the most returns in both groups (12,751 Control vs 16,065 Test), likely restarts or re-entries into the process.
- Returns to confirm are much higher in Test (6,340 vs 1,819).
- Raw counts are affected by group size (Test is larger), so the % table below is used for comparison.

In [14]:
# % of completed clients who repeated each step
step_return_pct = (completed[steps] > 1).groupby(completed["Variation"]).mean().mul(100).round(2).T
step_return_pct

Variation,Control,Test
start,37.28,38.23
step_1,28.05,32.78
step_2,24.29,19.41
step_3,19.46,13.20
confirm,6.80,19.85


% of completed clients who repeated each step
- Start is the most repeated step in both groups (~38%).
- Test reduces repeats at step_2 (19.41% vs 24.29%) and step_3 (13.20% vs 19.46%). These steps improved.
- Test increases repeats at step_1 (32.78% vs 28.05%) and especially at confirm (19.85% vs 6.80%, almost 3×).
- Repeated confirms may indicate double submissions, page reloads or unclear confirmation feedback in the new design. To be checked with the errors analysis.

In [15]:
#Where do people give up?
# journey_outcome covers the completed / non-completed rate

stuck_at_start...
Not completed: the furthest step reached before abandoning

confirm_but_skipped_step
Not completed: reached confirm, but missed a step

completed
All completers, perfect and with repeats together

In [16]:
reached = client_counts[steps].gt(0)   # True/False: did the client reach each step?

# Furthest step reached = the last True column, following the flow order
client_counts["furthest_step"] = reached.iloc[:, ::-1].idxmax(axis=1)

# One outcome per client
outcome = "stuck_at_" + client_counts["furthest_step"]
outcome = outcome.where(client_counts["furthest_step"] != "confirm", "confirm_but_skipped_step")
outcome = outcome.where(~client_counts["completed"], "completed")
client_counts["journey_outcome"] = outcome

In [17]:
order = ["stuck_at_start", "stuck_at_step_1", "stuck_at_step_2", "stuck_at_step_3",
         "confirm_but_skipped_step", "completed"]

stuck_counts = pd.crosstab(client_counts["journey_outcome"], client_counts["Variation"]).reindex(order)
stuck_counts

Variation,Control,Test
journey_outcome,,
stuck_at_start,3285,2454
stuck_at_step_1,1455,1980
stuck_at_step_2,1265,1411
stuck_at_step_3,2093,2436
confirm_but_skipped_step,228,349
completed,15206,18338


Journey outcome per client (counts)
- Every client is assigned exactly one outcome: the furthest step reached, "confirm but skipped a step", or "completed".
- Totals match the group sizes (23,532 Control, 26,968 Test), so no client was lost in the classification.
- Test has more completers (18,338 vs 15,206) and fewer clients stopping at start (2,454 vs 3,285), despite being the larger group.

In [18]:
stuck_pct = (pd.crosstab(client_counts["journey_outcome"], client_counts["Variation"],
                         normalize="columns")
               .mul(100).round(2).reindex(order))
stuck_pct

Variation,Control,Test
journey_outcome,,
stuck_at_start,13.96,9.10
stuck_at_step_1,6.18,7.34
stuck_at_step_2,5.38,5.23
stuck_at_step_3,8.89,9.03
confirm_but_skipped_step,0.97,1.29
completed,64.62,68.00


Where clients stop (% of all clients)
- Completion rate: 68.00% Test vs 64.62% Control (+3.4 pp).
- Start is the main bottleneck in Control** (13.96%). The new design reduces it to 9.10%, the clearest improvement.
- Step_1 got slightly worse in Test (7.34% vs 6.18%).
- Step_2 is similar in both groups (~5.3%).
- Step_3 remains a bottleneck in both (~9%): clients get close to the end and leave. The new design did not fix this step.

In [19]:
# Which step is mssing the most
skipped = client_counts[client_counts["journey_outcome"] == "confirm_but_skipped_step"]

(skipped[steps] == 0).groupby(skipped["Variation"]).sum().T

Variation,Control,Test
start,116,275
step_1,85,241
step_2,132,271
step_3,105,242
confirm,0,0


Clients who reached confirm with missing steps
- A small share of clients reached confirm without all steps recorded (0.97% Control vs 1.29% Test).
- Most of them are missing several steps, not just one. Many have only a confirm event, so their journey was not recorded.
- This is more frequent in Test, possibly due to tracking differences in the new design. It is treated as a data-quality limitation, not a UX finding.
- Decision: confirm-only clients are excluded from the analysis.

In [21]:
not_completed = client_counts[~client_counts["completed"]]

order = ["stuck_at_start", "stuck_at_step_1", "stuck_at_step_2", "stuck_at_step_3",
         "confirm_but_skipped_step"]

# Number of clients
stuck_nc_counts = pd.crosstab(not_completed["journey_outcome"],
                              not_completed["Variation"]).reindex(order)
stuck_nc_counts

Variation,Control,Test
journey_outcome,,
stuck_at_start,3285,2454
stuck_at_step_1,1455,1980
stuck_at_step_2,1265,1411
stuck_at_step_3,2093,2436
confirm_but_skipped_step,228,349


In [22]:
# % of non-completers in each group
stuck_nc_pct = (pd.crosstab(not_completed["journey_outcome"], not_completed["Variation"],
                            normalize="columns")
                  .mul(100).round(2).reindex(order))
stuck_nc_pct

Variation,Control,Test
journey_outcome,,
stuck_at_start,39.45,28.44
stuck_at_step_1,17.48,22.94
stuck_at_step_2,15.19,16.35
stuck_at_step_3,25.14,28.23
confirm_but_skipped_step,2.74,4.04


Where non-completers stop (% of non-completers)
- In Control, start is where most non-completers stop (39.45%). In Test this drops to 28.44%.
- In Test, a larger share stops at step_1 (22.94% vs 17.48%) and step_3 (28.23% vs 25.14%).
- The new design moves drop-off away from the start, but step_1 and step_3 now account for more of the abandonment.

In [ ]:
stage = {
    "stuck_at_start":  "early (start / step_1)",
    "stuck_at_step_1": "early (start / step_1)",
    "stuck_at_step_2": "late (step_2 / step_3)",
    "stuck_at_step_3": "late (step_2 / step_3)",
    "confirm_but_skipped_step": "reached confirm with a gap",
}
not_completed = not_completed.assign(dropoff_stage=not_completed["journey_outcome"].map(stage))

pd.crosstab(not_completed["dropoff_stage"], not_completed["Variation"],
            normalize="columns").mul(100).round(2)

Early vs late drop-off
- Most non-completers leave early (start / step_1) in both groups, but less so in Test: 51.38% vs 56.93%.
- In Test, drop-off shifts later (step_2 / step_3): 44.58% vs 40.33%.
- The new design keeps clients in the process longer, but many still leave before confirming, mainly at step_3.